# Parameter Dominace Comparison

## Imports

In [29]:
from pathlib import Path
from typing import Dict, List, Tuple, Any
import json
import math
import numpy as np
import pandas as pd
import importlib.util

In [41]:
ROOT = Path("../eval/system_outputs")
PROMPT = "bright_to_warm" # change for other prompts
INSTRUMENT = "piano" # or violin
METHOD_1 = "InstructFX2FX"
METHOD_2 = "LLM+LLM"
base_dir = ROOT / PROMPT / INSTRUMENT
exp_dirs = sorted([p for p in base_dir.iterdir() if p.is_dir() and p.name.startswith("experiment_")])

if not exp_dirs:
    raise FileNotFoundError(f"No experiment folders found under {base_dir}")

EXP_DIR = exp_dirs[-1]

sample_dirs = sorted([p for p in EXP_DIR.iterdir() if p.is_dir() and p.name.startswith(INSTRUMENT)])
if not sample_dirs:
    raise FileNotFoundError(f"No sample folders found under {EXP_DIR}")

SAMPLE_DIR = sample_dirs[0]

METHOD_DIR_1 = SAMPLE_DIR / METHOD_1
METHOD_DIR_2 = SAMPLE_DIR / METHOD_2

print("Using experiment folder:", EXP_DIR)
print("Using sample folder:", SAMPLE_DIR)
print("Using method 1 folder:", METHOD_DIR_1)
print("Using method 2 folder:", METHOD_DIR_2)
print("Method 1 exists:", METHOD_DIR_1.exists())
print("Method 2 exists:", METHOD_DIR_2.exists())

Using experiment folder: ../eval/system_outputs/bright_to_warm/piano/experiment_2026-03-16_03-36-54
Using sample folder: ../eval/system_outputs/bright_to_warm/piano/experiment_2026-03-16_03-36-54/piano
Using method 1 folder: ../eval/system_outputs/bright_to_warm/piano/experiment_2026-03-16_03-36-54/piano/InstructFX2FX
Using method 2 folder: ../eval/system_outputs/bright_to_warm/piano/experiment_2026-03-16_03-36-54/piano/LLM+LLM
Method 1 exists: True
Method 2 exists: True


### Load Param Config

In [42]:
FX_PY_PATH = Path("effects/fx.py")

if not FX_PY_PATH.exists():
    raise FileNotFoundError(f"Could not find fx.py at {FX_PY_PATH}")

spec = importlib.util.spec_from_file_location("fx_module", FX_PY_PATH)
fx_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fx_module)

ALL_PARAM_RANGES = fx_module.ALL_PARAM_RANGES

EQ_PARAM_CONFIG = ALL_PARAM_RANGES["EQ"]
print("Loaded EQ param config with", len(EQ_PARAM_CONFIG), "parameters")

Loaded EQ param config with 18 parameters


### Load Param Data

In [43]:
def load_json(json_path: Path) -> Any:
    with open(json_path, "r") as f:
        return json.load(f)

In [44]:
def get_eq_param_dict(data: Any) -> Dict[str, float]:
    if isinstance(data, dict) and "params" in data:
        params = data["params"]

        if isinstance(params, dict) and "EQ" in params and isinstance(params["EQ"], dict):
            return params["EQ"]

    if isinstance(data, dict) and "EQ" in data and isinstance(data["EQ"], dict):
        return data["EQ"]

    raise ValueError(
        f"Could not locate EQ parameter dict. "
        f"type={type(data)}, top-level keys={list(data.keys()) if isinstance(data, dict) else 'N/A'}"
    )

In [45]:
def extract_eq_param_names(json_path: Path) -> List[str]:
    data = load_json(json_path)
    eq_params = get_eq_param_dict(data)
    return list(eq_params.keys())

def flatten_eq_params(json_path: Path, param_names: List[str]) -> np.ndarray:
    data = load_json(json_path)
    eq_params = get_eq_param_dict(data)
    return np.array([eq_params[name] for name in param_names], dtype=float)

In [46]:
def normalize_single_value(value: float, lo: float, hi: float, scale: str) -> float:
    if scale == "linear":
        return (value - lo) / (hi - lo)

    if scale == "log":
        if value <= 0 or lo <= 0 or hi <= 0:
            raise ValueError(f"log-scale normalization requires positive values, got value={value}, lo={lo}, hi={hi}")
        return (math.log(value) - math.log(lo)) / (math.log(hi) - math.log(lo))

    raise ValueError(f"Unknown scale type: {scale}")

def normalize_eq_params(raw_params: np.ndarray, param_names: List[str]) -> np.ndarray:
    norm = []
    for value, name in zip(raw_params, param_names):
        cfg = EQ_PARAM_CONFIG[name]
        lo = cfg["lo"]
        hi = cfg["hi"]
        scale = cfg["scale"]
        norm.append(normalize_single_value(value, lo, hi, scale))
    return np.array(norm, dtype=float)

In [47]:
def load_instruct_final_tensor(run_dir: Path) -> np.ndarray:
    end_json = run_dir / "intermediate" / "optimization_steps" / "end_params.json"

    if not end_json.exists():
        raise FileNotFoundError(f"Missing end_params.json: {end_json}")

    data = load_json(end_json)

    if "params_tensor" not in data:
        raise ValueError(f"end_params.json does not contain 'params_tensor': {end_json}")

    arr = np.array(data["params_tensor"], dtype=float).squeeze()

    if arr.ndim != 1:
        raise ValueError(f"Expected 1D params tensor after squeeze, got shape {arr.shape} in {end_json}")

    return arr

In [48]:
def load_single_run_final_displacement(run_dir: Path, method_name: str) -> Tuple[np.ndarray, List[str]]:
    init_json = run_dir / "intermediate" / "01_initialized_params.json"

    if not init_json.exists():
        raise FileNotFoundError(f"Missing initialized json: {init_json}")

    param_names = extract_eq_param_names(init_json)

    init_raw = flatten_eq_params(init_json, param_names)
    init_norm = normalize_eq_params(init_raw, param_names)

    if method_name == "InstructFX2FX":
        final_norm = load_instruct_final_tensor(run_dir)

    elif method_name == "LLM+LLM":
        final_json = run_dir / "intermediate" / "02_refined_params.json"

        if not final_json.exists():
            raise FileNotFoundError(f"Missing refined json: {final_json}")

        final_raw = flatten_eq_params(final_json, param_names)
        final_norm = normalize_eq_params(final_raw, param_names)

    else:
        raise ValueError(f"Unsupported method name: {method_name}")

    if len(final_norm) != len(param_names):
        raise ValueError(
            f"Dimension mismatch: final_norm has {len(final_norm)} dims "
            f"but param_names has {len(param_names)} entries in {run_dir}"
        )

    delta = final_norm - init_norm
    return delta, param_names

In [49]:
def load_method_displacement_matrix(method_dir: Path, method_name: str) -> Tuple[np.ndarray, List[str], List[str]]:
    run_dirs = sorted([p for p in method_dir.iterdir() if p.is_dir() and p.name.startswith("run_")])

    if not run_dirs:
        raise RuntimeError(f"No run folders found in {method_dir}")

    deltas = []
    run_labels = []
    param_names = None

    for run_dir in run_dirs:
        try:
            delta, names = load_single_run_final_displacement(run_dir, method_name)
            deltas.append(delta)
            run_labels.append(run_dir.name)

            if param_names is None:
                param_names = names
            else:
                if names != param_names:
                    raise ValueError(f"Parameter name mismatch in {run_dir}")

        except Exception as e:
            print(f"[WARN] skipping {run_dir}: {e}")

    if not deltas:
        raise RuntimeError(f"No valid runs found in {method_dir}")

    X = np.stack(deltas, axis=0)   # shape (R, d)
    return X, param_names, run_labels

### PCA Setup

In [50]:
def movement_score(X: np.ndarray) -> np.ndarray:
    """
    Average absolute final displacement across runs.
    """
    return np.mean(np.abs(X), axis=0)


def zero_mean_matrix(X: np.ndarray) -> np.ndarray:
    return X - np.mean(X, axis=0, keepdims=True)


def run_pca(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    Xc = zero_mean_matrix(X)
    cov = (Xc.T @ Xc) / max(Xc.shape[0] - 1, 1)

    eigenvals, eigenvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigenvals)[::-1]

    eigenvals = eigenvals[idx]
    eigenvecs = eigenvecs[:, idx]

    total = np.sum(eigenvals)
    explained_ratio = eigenvals / total if total > 0 else np.zeros_like(eigenvals)

    return eigenvals, eigenvecs, explained_ratio


def compute_pca_contribution_score(
    eigenvecs: np.ndarray,
    explained_ratio: np.ndarray,
    n_components: int = 3
) -> np.ndarray:
    K = min(n_components, eigenvecs.shape[1])
    weights = explained_ratio[:K]
    vecs = eigenvecs[:, :K]
    return np.sum((vecs ** 2) * weights[np.newaxis, :], axis=1)


def compute_combined_score(
    movement_score_arr: np.ndarray,
    pca_score_arr: np.ndarray
) -> np.ndarray:
    return movement_score_arr * pca_score_arr


def compute_sensitivity_candidate_score(
    movement_score_arr: np.ndarray,
    pca_score_arr: np.ndarray,
    eps: float = 1e-8
) -> np.ndarray:
    return pca_score_arr / (movement_score_arr + eps)


def rank_parameters(
    param_names: List[str],
    movement_score_arr: np.ndarray,
    pca_score_arr: np.ndarray,
    dominance_score_arr: np.ndarray,
    sensitivity_score_arr: np.ndarray
) -> pd.DataFrame:
    return pd.DataFrame({
        "parameter": param_names,
        "movement_score": movement_score_arr,
        "pca_score": pca_score_arr,
        "dominance_score": dominance_score_arr,
        "sensitivity_candidate_score": sensitivity_score_arr,
    })


def analyze_method_final_displacement(
    method_dir: Path,
    method_name: str,
    n_components: int = 3
) -> pd.DataFrame:
    X, param_names, run_labels = load_method_displacement_matrix(method_dir, method_name)

    move_score = movement_score(X)
    eigenvals, eigenvecs, explained_ratio = run_pca(X)
    pca_score_arr = compute_pca_contribution_score(
        eigenvecs,
        explained_ratio,
        n_components=n_components
    )
    dominance_score_arr = compute_combined_score(move_score, pca_score_arr)
    sensitivity_score_arr = compute_sensitivity_candidate_score(move_score, pca_score_arr)

    ranking = rank_parameters(
        param_names,
        move_score,
        pca_score_arr,
        dominance_score_arr,
        sensitivity_score_arr
    )
    return ranking

### Display Comparison

In [51]:
ranking_1 = analyze_method_final_displacement(METHOD_DIR_1, METHOD_1, n_components=3)
ranking_2 = analyze_method_final_displacement(METHOD_DIR_2, METHOD_2, n_components=3)

compare_basic = ranking_1.merge(
    ranking_2,
    on="parameter",
    suffixes=("_instructfx2fx", "_llm_llm")
)

compare_basic = compare_basic.rename(columns={
    "movement_score_instructfx2fx": "instructfx2fx_movement",
    "movement_score_llm_llm": "llm_llm_movement",
    "pca_score_instructfx2fx": "instructfx2fx_pca",
    "pca_score_llm_llm": "llm_llm_pca",
})

compare_basic = compare_basic[
    [
        "parameter",
        "instructfx2fx_movement",
        "llm_llm_movement",
        "instructfx2fx_pca",
        "llm_llm_pca",
    ]
]

display(compare_basic)

params_instructfx2fx = (
    ranking_1
    .sort_values("pca_score", ascending=False)["parameter"]
    .tolist()
)

params_llm_llm = (
    ranking_2
    .sort_values("pca_score", ascending=False)["parameter"]
    .tolist()
)

print("InstructFX2FX (by PCA score):")
print(params_instructfx2fx)
print("\nLLM+LLM (by PCA score):")
print(params_llm_llm)

,parameter,instructfx2fx_movement,llm_llm_movement,instructfx2fx_pca,llm_llm_pca
0,b1_freq,0.176351,0.005017,0.046632,0.023147
1,b1_gain,0.137351,0.045833,0.043734,0.019033
2,b1_q,0.305349,0.026534,0.068218,0.087095
3,b2_freq,0.267186,0.005017,0.083542,0.018650
4,b2_gain,0.132787,0.032292,0.017579,0.013264
5,b2_q,0.333412,0.022267,0.229908,0.102770
6,b3_freq,0.117588,0.005313,0.001075,0.019922
7,b3_gain,0.075979,0.020313,0.000387,0.030846
8,b3_q,0.447203,0.023974,0.003187,0.111649
9,b4_freq,0.057428,0.004255,0.006944,0.025523


InstructFX2FX (by PCA score):
['b2_q', 'b5_freq', 'b2_freq', 'b5_q', 'b1_q', 'b6_q', 'b5_gain', 'b1_freq', 'b1_gain', 'b4_gain', 'b2_gain', 'b4_freq', 'b4_q', 'b3_q', 'b6_freq', 'b3_freq', 'b6_gain', 'b3_gain']

LLM+LLM (by PCA score):
['b3_q', 'b2_q', 'b1_q', 'b6_q', 'b4_q', 'b3_gain', 'b5_q', 'b6_gain', 'b4_freq', 'b1_freq', 'b5_freq', 'b3_freq', 'b1_gain', 'b2_freq', 'b5_gain', 'b2_gain', 'b6_freq', 'b4_gain']
